# ExpenseFlow Export EDA — Inference Baseline

**Purpose:** this notebook is exploratory only. It loads the full 2,000,000-row
generated export using ordinary eager `pandas` loading with type inference left
on its defaults, so we can observe exactly what inference does to the raw data
before Task 2 replaces it with an explicit declared schema.

**This notebook does not clean, repair, or fix anything.** Every issue found
here is left exactly as inference produces it, and is called out explicitly in
the final comparison section.

Input file: `data/exports/expense_export_seed42_rows2000000.jsonl.gz`
(seed=42, rows=2,000,000 — generated by `services/pipeline/tools/generate_export.py`)


In [ ]:
import gzip
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

EXPORT_PATH = Path("../data/exports/expense_export_seed42_rows2000000.jsonl.gz")
assert EXPORT_PATH.exists(), f"missing export file: {EXPORT_PATH}"
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)
print("file path:", EXPORT_PATH.resolve())
print("file size (bytes):", EXPORT_PATH.stat().st_size)


## 1. Load with ordinary eager inference

`pd.read_json(..., lines=True)` with no `dtype=` argument, no `convert_dates=`
override, no explicit schema of any kind — this is what a naive first pass at
loading this file looks like, and it is intentionally the default/naive path.


In [ ]:
start = time.perf_counter()
with gzip.open(EXPORT_PATH, "rt", encoding="utf-8") as f:
    df = pd.read_json(f, lines=True)
elapsed = time.perf_counter() - start
print(f"loaded in {elapsed:.2f}s")


## 2. Exact row count and column count

In [ ]:
n_rows, n_cols = df.shape
print("row count:", n_rows)
print("column count:", n_cols)
print("columns:", list(df.columns))


## 3. Inferred dtype of every column

In [ ]:
dtypes_report = df.dtypes.to_frame(name="inferred_dtype")
dtypes_report


## 4. Null count and null percentage for every column

In [ ]:
null_counts = df.isnull().sum()
null_pct = (null_counts / n_rows * 100).round(3)
null_report = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
null_report


## 5. Distinct-value counts for every identifier-like column

Identifier-like columns: `tenant_id`, `expense_report_id`, `record_id`,
`submitter_id`, `receipt_number`.


In [ ]:
identifier_cols = ["tenant_id", "expense_report_id", "record_id", "submitter_id", "receipt_number"]
identifier_nunique = pd.DataFrame({
    "distinct_count": [df[c].nunique(dropna=True) for c in identifier_cols],
    "non_null_count": [df[c].notnull().sum() for c in identifier_cols],
}, index=identifier_cols)
identifier_nunique["duplicate_non_null_rows"] = (
    identifier_nunique["non_null_count"] - identifier_nunique["distinct_count"]
)
identifier_nunique


## 6. Distinct-value counts for GL-code columns

GL-code-related columns: `gl_account_code`, `gl_account_name`,
`gl_normal_balance`, `gl_coding_status`.


In [ ]:
gl_cols = ["gl_account_code", "gl_account_name", "gl_normal_balance", "gl_coding_status"]
gl_nunique = pd.DataFrame({
    "distinct_count": [df[c].nunique(dropna=True) for c in gl_cols],
    "non_null_count": [df[c].notnull().sum() for c in gl_cols],
}, index=gl_cols)
gl_nunique


In [ ]:
print("distinct raw gl_account_code values (sorted):")
sorted(df["gl_account_code"].dropna().unique().tolist())


## 7. Minimum and maximum for every numeric (inferred) column

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("columns pandas inferred as numeric:", numeric_cols)
numeric_minmax = pd.DataFrame({
    "min": [df[c].min() for c in numeric_cols],
    "max": [df[c].max() for c in numeric_cols],
    "dtype": [df[c].dtype for c in numeric_cols],
}, index=numeric_cols)
numeric_minmax


Note: `miles` is deliberately **excluded** from the block above because
pandas inferred it as `object` (string), not numeric — see the dtype table in
section 3 and the raw-vs-inferred comparison at the end. It is profiled
separately below as a string column.


In [ ]:
print("miles column dtype:", df["miles"].dtype)
non_null_miles = df["miles"].dropna()
print("miles non-null count:", len(non_null_miles))
print("miles sample raw values:", non_null_miles.head(5).tolist())
# Cast only for observation purposes — this cast is NOT part of the loaded
# dataframe `df`, it is a throwaway Series used only to report a numeric
# min/max for comparison. df itself is left untouched (object dtype).
miles_as_float = non_null_miles.astype(float)
print("miles min (via manual float cast, for observation only):", miles_as_float.min())
print("miles max (via manual float cast, for observation only):", miles_as_float.max())


## 8. Value/frequency distributions for categorical-style columns

In [ ]:
categorical_cols = [
    "record_type",
    "current_stage",
    "category",
    "currency",
    "gl_normal_balance",
    "gl_coding_status",
    "manager_review_status",
    "flagged",
    "deductible",
]
for col in categorical_cols:
    print(f"--- {col} (dtype={df[col].dtype}) ---")
    print(df[col].value_counts(dropna=False))
    print()


## 9. Distinct date formats / parsing patterns present in date columns

`created_at`, `receipt_date`, and `trip_date` were all loaded as plain
`object` (string) columns by inference — `pd.read_json` does not auto-parse
ISO date/datetime strings into `datetime64` dtype by default. We inspect the
raw string patterns actually present using regex classification, without
converting the column.


In [ ]:
import re

ISO_DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
ISO_DATETIME_RE = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$")
US_DATE_RE = re.compile(r"^\d{2}/\d{2}/\d{4}$")


def classify_date_string(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "null"
    if ISO_DATETIME_RE.match(value):
        return "iso_datetime (YYYY-MM-DDTHH:MM:SSZ)"
    if ISO_DATE_RE.match(value):
        return "iso_date (YYYY-MM-DD)"
    if US_DATE_RE.match(value):
        return "us_date (MM/DD/YYYY)"
    return "other/unrecognized"


date_like_cols = ["created_at", "receipt_date", "trip_date"]
for col in date_like_cols:
    print(f"--- {col} (dtype={df[col].dtype}) format classification ---")
    pattern_counts = df[col].apply(classify_date_string).value_counts(dropna=False)
    print(pattern_counts)
    print()


In [ ]:
print("trip_date: sample of us_date-format rows found by inference-free regex scan:")
trip_date_patterns = df["trip_date"].apply(classify_date_string)
us_format_sample = df.loc[trip_date_patterns == "us_date (MM/DD/YYYY)", ["record_id", "record_type", "trip_date"]].head(5)
us_format_sample


## 10. Length distribution for every identifier-like column

Computed on the raw string values as loaded (before any casting).


In [ ]:
length_stats = {}
for col in identifier_cols + ["gl_account_code"]:
    lengths = df[col].dropna().astype(str).str.len()
    length_stats[col] = {
        "min_len": lengths.min(),
        "max_len": lengths.max(),
        "mean_len": round(lengths.mean(), 2),
        "distinct_lengths": sorted(lengths.unique().tolist()),
    }
pd.DataFrame(length_stats).T


## 11. Eager dataframe memory usage at full scale

`memory_usage(deep=True)` is used because shallow memory usage on `object`
columns under-reports actual Python string memory by roughly an order of
magnitude — deep=True walks the actual Python objects.


In [ ]:
mem_per_col = df.memory_usage(deep=True)
mem_report = mem_per_col.to_frame(name="bytes")
mem_report["MB"] = (mem_report["bytes"] / (1024 ** 2)).round(2)
mem_report.loc["TOTAL"] = mem_report.sum()
mem_report


In [ ]:
total_mb = mem_per_col.sum() / (1024 ** 2)
total_gb = total_mb / 1024
print(f"Total in-memory dataframe size: {total_mb:,.1f} MB ({total_gb:.2f} GB)")
print(f"Compressed file size on disk: {EXPORT_PATH.stat().st_size / (1024**2):,.1f} MB")
print(f"In-memory expansion factor over compressed file: {total_mb / (EXPORT_PATH.stat().st_size / (1024**2)):.1f}x")


## 12. Counts of every anomaly present in the generated export

These are the six intentional defect types from the agreed generator spec,
counted here directly against the *inferred* dataframe exactly as loaded (no
correction applied).


In [ ]:
anomaly_counts = {}

# Defect 1: lowercase currency (should always be "USD" or null)
anomaly_counts["lowercase_currency"] = int(
    df["currency"].dropna().apply(lambda v: v != v.upper()).sum()
)

# Defect 2: GL account code leading zero or trailing whitespace
anomaly_counts["gl_account_code_leading_zero_or_padding"] = int(
    df["gl_account_code"].dropna().apply(
        lambda v: v.startswith("0") or v != v.strip()
    ).sum()
)

# Defect 3: non-positive amount_cents (<=0), among non-null values
anomaly_counts["amount_cents_non_positive"] = int((df["amount_cents"].dropna() <= 0).sum())

# Defect 4: receipt_number populated but receipt_date null (line_item rows only)
line_items = df[df["record_type"] == "line_item"]
anomaly_counts["receipt_present_but_receipt_date_null"] = int(
    (line_items["receipt_number"].notnull() & line_items["receipt_date"].isnull()).sum()
)

# Defect 5: trip_date in MM/DD/YYYY format instead of ISO
trip_date_patterns = df["trip_date"].apply(classify_date_string)
anomaly_counts["trip_date_us_format"] = int((trip_date_patterns == "us_date (MM/DD/YYYY)").sum())

# Defect 6: duplicate record_id on consecutive rows (as generated; row order
# in df matches file order since read_json with lines=True preserves order)
record_ids = df["record_id"]
anomaly_counts["duplicate_record_id_consecutive_rows"] = int((record_ids == record_ids.shift(1)).sum())

anomaly_df = pd.Series(anomaly_counts, name="count").to_frame()
anomaly_df["pct_of_rows"] = (anomaly_df["count"] / n_rows * 100).round(4)
anomaly_df


## 13. Critical comparison: raw observations vs. inferred dataframe

Direct comparison against the raw-text inspection performed earlier (reading
the same file with `zcat`/`grep`, no parser involved). Each row below states
what the raw bytes actually contained vs. what eager `pd.read_json` inference
turned it into, and whether meaning was altered or lost.


In [ ]:
comparison_rows = [
    {
        "column": "gl_account_code",
        "raw_observation": 'JSON string, e.g. "6100" or defect "06100" / "6400 "',
        "inferred_dtype": str(df["gl_account_code"].dtype),
        "inference_changed_meaning": "NO — stayed object/string",
        "detail": (
            "pandas correctly kept this as object dtype because it's a JSON string "
            "and some values contain non-numeric characters (trailing space) or are "
            "not uniformly numeric-looking across the column. Leading zeros in "
            "\"06100\" are preserved exactly as-is. This is the lucky case, not a "
            "designed-in guarantee — a column with ONLY clean numeric-looking strings "
            "would have been at risk (see amount-like risk note below)."
        ),
    },
    {
        "column": "amount_cents",
        "raw_observation": "bare JSON integer, e.g. 224716, or 0/-50882 (defect), or null",
        "inferred_dtype": str(df["amount_cents"].dtype),
        "inference_changed_meaning": "PARTIAL — became float64, not int",
        "detail": (
            "Because the column contains nulls (mileage rows never have "
            "amount_cents), pandas cannot represent it as int64 (no native NaN for "
            "int64) and silently upcasts the entire column to float64. Money that "
            "was emitted as strict integer minor units in the file is now stored as "
            "floating point in memory — exactly the kind of silent representational "
            "change this notebook is meant to expose. No precision is lost yet at "
            "these magnitudes, but the type contract (integer cents) is broken."
        ),
    },
    {
        "column": "miles",
        "raw_observation": 'JSON string, e.g. "117.64", always 2 decimal places, or null',
        "inferred_dtype": str(df["miles"].dtype),
        "inference_changed_meaning": "NO type change, but NOT usable as numeric",
        "detail": (
            "pandas left this as object/string because it was written as a quoted "
            "JSON string in the source file (matching the DB's numeric(10,2) "
            "semantics deliberately). Inference does NOT auto-promote it to float, "
            "so any numeric aggregation (sum/mean/min/max) on this column will fail "
            "or misbehave until it is explicitly cast — this is a case where "
            "inference under-processes a column that a schema step will need to "
            "handle explicitly (Decimal-safe cast, not naive float)."
        ),
    },
    {
        "column": "created_at / receipt_date / trip_date",
        "raw_observation": (
            "ISO date/datetime strings, PLUS a seeded MM/DD/YYYY variant on "
            "trip_date, plus null"
        ),
        "inferred_dtype": (
            f'{df["created_at"].dtype} / {df["receipt_date"].dtype} / {df["trip_date"].dtype}'
        ),
        "inference_changed_meaning": "NO — remained generic object/string",
        "detail": (
            "pd.read_json does NOT auto-parse date-shaped strings into datetime64 "
            "by default (unlike read_csv with parse_dates, or convert_dates=True "
            "which is read_json's own default for keys literally named like "
            "'_at'/'_time' patterns in some pandas versions — verified here it did "
            "NOT trigger). All three date columns remained plain strings. This is "
            "actually the SAFE outcome for trip_date, because it prevented "
            "pandas/numpy from silently guessing a datetime parse for the mixed "
            "ISO / MM-DD-YYYY formats (which could have misinterpreted ambiguous "
            "dates, e.g. 03/04/2025). But it also means downstream code cannot do "
            "date arithmetic without an explicit, format-aware parse step — "
            "inference left the ambiguity for a human/schema to resolve rather "
            "than silently guessing wrong."
        ),
    },
    {
        "column": "tenant_id / expense_report_id / record_id",
        "raw_observation": "UUID-shaped JSON strings, hyphenated hex",
        "inferred_dtype": (
            f'{df["tenant_id"].dtype} / {df["expense_report_id"].dtype} / {df["record_id"].dtype}'
        ),
        "inference_changed_meaning": "NO — stayed object/string",
        "detail": (
            "UUIDs contain hyphens and non-digit hex characters, so pandas has no "
            "opportunity to infer them numerically. They are preserved as exact "
            "strings. Flagged here as a NEGATIVE finding (i.e. no problem found) "
            "specifically because the task asked us to check for identifiers "
            "inferred numerically — this class of identifier is naturally immune, "
            "unlike gl_account_code or receipt_number, which are numeric-LOOKING "
            "strings and were at real risk."
        ),
    },
    {
        "column": "receipt_number",
        "raw_observation": 'JSON string, e.g. "RCT-205907", or null',
        "inferred_dtype": str(df["receipt_number"].dtype),
        "inference_changed_meaning": "NO — stayed object/string",
        "detail": (
            "The RCT- prefix protects this column from numeric inference. If this "
            "identifier were ever emitted as a bare digit string with no prefix "
            "(e.g. \"205907\"), it would be a strong candidate for accidental "
            "numeric inference and leading-zero loss under a stricter loader "
            "(e.g. read_csv with default dtype inference) — worth flagging as a "
            "latent risk even though this export's prefix convention avoided it "
            "this time."
        ),
    },
    {
        "column": "flagged / deductible",
        "raw_observation": "JSON boolean literals true/false, never null",
        "inferred_dtype": f'{df["flagged"].dtype} / {df["deductible"].dtype}',
        "inference_changed_meaning": "NO — correctly inferred as bool",
        "detail": "Straightforward, no nulls present on these two columns so no float upcast risk applied here.",
    },
    {
        "column": "merchant / origin / destination / business_purpose",
        "raw_observation": "JSON string or null depending on record_type",
        "inferred_dtype": (
            f'{df["merchant"].dtype} / {df["origin"].dtype} / '
            f'{df["destination"].dtype} / {df["business_purpose"].dtype}'
        ),
        "inference_changed_meaning": "NO — stayed object/string with NaN for null",
        "detail": (
            "Structural nulls (fields that only apply to one record_type) are "
            "represented as NaN in the dataframe. This is unremarkable for string "
            "columns (no dtype-forcing side effect), unlike the numeric column "
            "case below."
        ),
    },
]

pd.set_option("display.max_colwidth", None)
comparison_df = pd.DataFrame(comparison_rows)
comparison_df


## 14. Nulls forcing an unexpected dtype — explicit isolation

Directly demonstrating the `amount_cents` int→float upcast referenced above,
since it is the single clearest case of inference silently changing the
*meaning* (not just the display) of the raw data.


In [ ]:
print("amount_cents dtype in loaded dataframe:", df["amount_cents"].dtype)
print("amount_cents contains nulls:", df["amount_cents"].isnull().any())
print("amount_cents null count:", df["amount_cents"].isnull().sum(), f"({df['amount_cents'].isnull().sum() / n_rows * 100:.2f}% of rows)")
print()
print("sample of amount_cents values as actually stored in the dataframe (note the trailing .0):")
df["amount_cents"].dropna().head(5)


## 15. Summary of observed facts

This is a factual summary only — nothing here is a recommendation or a fix.


In [ ]:
summary_lines = [
    f"Row count: {n_rows:,}",
    f"Column count: {n_cols}",
    "",
    "Inferred dtypes:",
]
for col, dtype in df.dtypes.items():
    summary_lines.append(f"  - {col}: {dtype}")

summary_lines += [
    "",
    f"Total eager in-memory size (deep): {total_mb:,.1f} MB ({total_gb:.2f} GB)",
    f"Compressed file size on disk: {EXPORT_PATH.stat().st_size / (1024**2):,.1f} MB",
    "",
    "Anomaly counts (from section 12):",
]
for k, v in anomaly_counts.items():
    summary_lines.append(f"  - {k}: {v}")

summary_lines += [
    "",
    "Raw-vs-inferred problems found (from section 13):",
    "  - amount_cents: int in file -> float64 in dataframe (null-forced upcast); integer-minor-units contract broken",
    "  - miles: string in file -> stays object/string in dataframe; NOT usable as numeric without an explicit cast",
    "  - created_at / receipt_date / trip_date: all remain generic object/string; no dtype corruption, but no usable date semantics either, and the mixed ISO/MM-DD-YYYY formats in trip_date are invisible unless specifically searched for",
    "  - gl_account_code: leading zeros and trailing whitespace defects survived intact in this run (object dtype), but only because the column wasn't uniformly clean numeric-looking text — flagged as a fragile, not guaranteed, outcome",
    "  - identifiers (tenant_id/expense_report_id/record_id/receipt_number): no numeric misinference observed, protected by non-digit characters (hyphens) or a text prefix (RCT-)",
]

print("\n".join(summary_lines))
